# 03 — Correlation Analysis (7 Charts)

**Purpose**: Core correlation notebook. Addresses rebuttal argument 2 (correlations rise during crises)
and argument 6 (Liv-ex indices are misleading benchmarks).

**Inputs** (from `01_data_setup.ipynb`):
- `data/livex_indices_clean.parquet`
- `data/comparison_assets_monthly.parquet`

**Outputs** — 7 PNGs in `images/correlation/`:
1. `01_static_correlation_heatmap.png`
2. `02_rolling_correlations_vs_sp500.png`
3. `03_lx100_rolling_correlation_all_assets.png`
4. `04_gfc_drawdown_comparison.png`
4b. `04b_covid_multiwindow_comparison.png`
5. `05_burgundy150_vs_sp500_ftse_2008.png`
6. `06_burgundy150_gfc_bar_comparison.png`

**All prices are in GBP unless otherwise stated.**

## 1. Environment Setup

In [1]:
import sys
from pathlib import Path

# ---------------------------------------------------------------------------
# Ensure the repo root is in sys.path so project modules can be imported
# regardless of whether this notebook is opened from the repo root or the
# notebook directory itself.
# ---------------------------------------------------------------------------
def _find_repo_root(start: Path) -> Path:
    for parent in [start, *start.parents]:
        if (parent / 'pyproject.toml').exists() or (parent / '.git').exists():
            return parent
    raise RuntimeError(
        'Could not find repo root (no pyproject.toml or .git found). '
        f'Searched from: {start}'
    )

_repo_root = str(_find_repo_root(Path.cwd()))
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

print(f'Repo root: {_repo_root}')

Repo root: /home/runner/work/report-for-me/report-for-me


In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import yfinance as yf
from pathlib import Path

warnings.filterwarnings("ignore")

# ── Paths ──────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path("__file__").resolve().parent
PROJECT_DIR  = NOTEBOOK_DIR.parent
DATA_DIR     = PROJECT_DIR / "data"
IMAGES_DIR   = PROJECT_DIR / "images" / "correlation"
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

LIVEX_PARQUET      = DATA_DIR / "livex_indices_clean.parquet"
COMPARISON_PARQUET = DATA_DIR / "comparison_assets_monthly.parquet"

# ── WineFi colour palette ──────────────────────────────────────────────────
WINEFI_COLOURS = [
    '#9437ff',  # purple  — primary
    '#83D483',  # mantis
    '#FFD166',  # sunglow
    '#F78C6B',  # coral
    '#4D87D0',  # blue
    '#EF476F',  # red
    '#06D6A0',  # emerald
    '#C23FB7',  # pink/purple
    '#4A4A68',  # slate
]

# Named aliases mapped to WineFi palette
CLARET    = WINEFI_COLOURS[0]   # purple  — Liv-ex primary
ROSE      = WINEFI_COLOURS[1]   # mantis  — Burgundy 150
CHAMPAGNE = WINEFI_COLOURS[2]   # sunglow — gold accent
NAVY      = WINEFI_COLOURS[4]   # blue    — equities (S&P 500)
FOREST    = WINEFI_COLOURS[6]   # emerald — FTSE 100
MIDNIGHT  = '#2C2C2C'           # near-black for text
WARM_GREY = '#9E9E9E'           # subdued / caption text

COLOURS = {
    "Liv-ex Fine Wine 100":  CLARET,
    "Liv-ex Fine Wine 1000": WINEFI_COLOURS[3],  # coral — LX1000
    "Burgundy 150":          ROSE,
    "sp500":                 NAVY,
    "sp500_gbp":             NAVY,
    "ftse100":               FOREST,
    "gold":                  CHAMPAGNE,
}

LABELS = {
    "Liv-ex Fine Wine 100":  "Liv-ex 100",
    "Liv-ex Fine Wine 1000": "Liv-ex 1000",
    "Burgundy 150":          "Burgundy 150",
    "sp500":                 "S&P 500 TR (USD)",
    "sp500_gbp":             "S&P 500 TR (GBP)",
    "ftse100":               "FTSE 100 TR",
    "gold":                  "Gold (USD)",
}

DPI = 150

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "grid.linestyle":   "--",
    "font.size":        10,
    "axes.titlesize":   12,
    "axes.labelsize":   10,
})

# Crisis periods for shading (used across multiple charts)
CRISES = [
    (pd.Timestamp("2000-03-01"), pd.Timestamp("2002-10-31"), "Dot-com"),
    (pd.Timestamp("2007-10-01"), pd.Timestamp("2009-03-31"), "GFC"),
    (pd.Timestamp("2020-02-01"), pd.Timestamp("2020-04-30"), "COVID"),
]

print("Setup complete.")
print(f"Images dir: {IMAGES_DIR}")
print(f"Parquets exist: livex={LIVEX_PARQUET.exists()}, comparison={COMPARISON_PARQUET.exists()}")

Setup complete.
Images dir: /home/runner/work/report-for-me/report-for-me/projects/correlation-diversification/images/correlation
Parquets exist: livex=False, comparison=False


## 2. Data Loading

Load parquets from `01_data_setup`. Falls back to CSV + yfinance if parquets are not yet available.

In [3]:
def _load_livex_from_csv():
    """Build monthly Liv-ex data directly from CSV (fallback when parquets absent)."""
    csv_path = DATA_DIR / "liv-ex_index_history.csv"
    if not csv_path.exists():
        raise FileNotFoundError(f"Liv-ex CSV not found at {csv_path}")
    try:
        df = pd.read_csv(csv_path, index_col="date", parse_dates=True)
        if df.index.dtype == "object":
            raise ValueError("date column not parsed as datetime")
    except (ValueError, KeyError):
        df = pd.read_csv(csv_path, index_col=0, parse_dates=True)
    df = df.select_dtypes(include="number")
    df.index = pd.to_datetime(df.index)
    df = df[df.index >= "2000-01-01"].sort_index()
    return df.resample("ME").last()


def _load_comparison_from_yfinance():
    """Build comparison asset prices from yfinance (fallback when parquets absent)."""
    tickers = {
        "sp500":   "^SP500TR",
        "ftse100": "ISF.L",
        "gold":    "GC=F",
        "gbpusd":  "GBPUSD=X",
    }
    frames = {}
    for name, ticker in tickers.items():
        raw = yf.download(ticker, start="2000-01-01", progress=False, auto_adjust=False)["Close"]
        if isinstance(raw, pd.DataFrame):
            raw = raw.squeeze()
        frames[name] = raw.resample("ME").last()
    return pd.DataFrame(frames)


# ── Load or build combined price DataFrame ────────────────────────────────
if COMPARISON_PARQUET.exists():
    print(f"Loading comparison parquet: {COMPARISON_PARQUET}")
    combined = pd.read_parquet(COMPARISON_PARQUET)
    combined.index = pd.to_datetime(combined.index)

    # Supplement with livex parquet if any columns are missing
    if LIVEX_PARQUET.exists():
        livex_df = pd.read_parquet(LIVEX_PARQUET)
        livex_df.index = pd.to_datetime(livex_df.index)
        for col in livex_df.columns:
            if col not in combined.columns:
                combined[col] = livex_df[col]

    # Fetch GBPUSD for GBP-adjusted S&P 500
    print("Downloading GBPUSD=X for S&P 500 GBP adjustment...")
    fx_raw = yf.download("GBPUSD=X", start="2000-01-01", progress=False, auto_adjust=False)["Close"]
    if isinstance(fx_raw, pd.DataFrame):
        fx_raw = fx_raw.squeeze()
    gbpusd_monthly = fx_raw.resample("ME").last()
    gbpusd_monthly.index = pd.to_datetime(gbpusd_monthly.index)
    combined = combined.join(gbpusd_monthly.rename("gbpusd"), how="left")

else:
    print("Parquets not found — building from CSV + yfinance (fallback mode)")
    livex_monthly = _load_livex_from_csv()
    comp_df       = _load_comparison_from_yfinance()   # includes gbpusd
    combined      = comp_df.join(livex_monthly, how="outer")

# ── Common post-load processing ───────────────────────────────────────────
combined = combined[combined.index >= "2000-01-01"].sort_index()

# Compute S&P 500 GBP-adjusted price: sp500_usd / gbpusd
if "gbpusd" in combined.columns and "sp500" in combined.columns:
    combined["sp500_gbp"] = combined["sp500"] / combined["gbpusd"]
elif "sp500" in combined.columns:
    print("WARNING: GBPUSD not available; sp500_gbp will equal sp500 (USD as proxy)")
    combined["sp500_gbp"] = combined["sp500"]

# Identify Liv-ex index columns (everything not a comparison asset or FX rate)
_non_livex = {"sp500", "ftse100", "gold", "gbpusd", "sp500_gbp"}
LIVEX_COLS = [
    c for c in combined.columns
    if c not in _non_livex and combined[c].notna().sum() > 12
]

# Detect Liv-ex 100 column
LX100_COL = None
for _candidate in ["Liv-ex Fine Wine 100", "Livex Fine Wine 100", "LX100"]:
    if _candidate in combined.columns:
        LX100_COL = _candidate
        break
if LX100_COL is None:
    _lx100_cands = [c for c in LIVEX_COLS if "100" in c and "fine" in c.lower()]
    if _lx100_cands:
        LX100_COL = _lx100_cands[0]
    elif LIVEX_COLS:
        LX100_COL = LIVEX_COLS[0]

# Detect Liv-ex 1000 column
LX1000_COL = None
for _candidate in ["Liv-ex Fine Wine 1000", "Livex Fine Wine 1000", "LX1000"]:
    if _candidate in combined.columns:
        LX1000_COL = _candidate
        break
if LX1000_COL is None:
    _lx1000_cands = [c for c in LIVEX_COLS if "1000" in c and "fine" in c.lower()]
    if _lx1000_cands:
        LX1000_COL = _lx1000_cands[0]

# Check Burgundy 150 availability
BURGUNDY_COL   = "Burgundy 150" if "Burgundy 150" in combined.columns else None
BURGUNDY_AVAIL = BURGUNDY_COL is not None and combined[BURGUNDY_COL].notna().sum() > 12

print(f"\nDataset shape:      {combined.shape}")
print(f"Date range:         {combined.index.min().date()} → {combined.index.max().date()}")
print(f"Liv-ex columns:     {LIVEX_COLS}")
print(f"LX100 column:       {LX100_COL}")
print(f"LX1000 column:      {LX1000_COL}")
print(f"Burgundy 150 avail: {BURGUNDY_AVAIL}")
combined[[c for c in ["sp500", "ftse100", "gold", "sp500_gbp"] if c in combined.columns]].tail(3)

Parquets not found — building from CSV + yfinance (fallback mode)



Dataset shape:      (315, 16)
Date range:         2000-01-31 → 2026-03-31
Liv-ex columns:     ['Burgundy 150', 'California 50', 'Champagne 50', 'Italy 100', 'Liv-ex Bordeaux 500', 'Liv-ex Fine Wine 100', 'Liv-ex Fine Wine 1000', 'Liv-ex Fine Wine 50', 'Port 50', 'Rest of the World 60', 'Rhone 100']
LX100 column:       Liv-ex Fine Wine 100
LX1000 column:      Liv-ex Fine Wine 1000
Burgundy 150 avail: True


,sp500,ftse100,gold,sp500_gbp
Date,,,,
2026-01-31,15441.150391,996.599976,4713.899902,11184.025326
2026-02-28,15323.799805,1067.599976,5230.500000,11358.153849
2026-03-31,14936.370117,1017.000000,5018.000000,11204.936029


## 3. Compute Monthly Log-Returns

In [4]:
# Exclude raw FX rate from return computation
_price_cols = [c for c in combined.columns if c != "gbpusd"]
returns = np.log(combined[_price_cols] / combined[_price_cols].shift(1)).dropna(how="all")

print(f"Returns shape: {returns.shape}")
print(f"Date range:    {returns.index.min().date()} → {returns.index.max().date()}")
returns[[c for c in [LX100_COL, "sp500_gbp", "ftse100", "gold"] if c in returns.columns]].describe().round(4)

Returns shape: (314, 15)
Date range:    2000-02-29 → 2026-03-31


,Liv-ex Fine Wine 100,sp500_gbp,ftse100,gold
count,295.0000,267.0000,206.0000,307.0000
mean,0.0042,0.0094,0.0043,0.0094
std,0.0230,0.0375,0.0365,0.0469
min,-0.1677,-0.0932,-0.1584,-0.1985
25%,-0.0056,-0.0121,-0.0191,-0.0203
50%,0.0034,0.0106,0.0085,0.0090
75%,0.0130,0.0360,0.0277,0.0396
max,0.1084,0.1126,0.1171,0.1299


## 4. Charts

### Chart 1 — Static Correlation Heatmap

Seaborn heatmap: all Liv-ex indices vs S&P 500, FTSE 100, Gold (monthly log-return correlations, full history).

In [5]:
CORR_ASSETS   = [c for c in ["sp500", "ftse100", "gold"] if c in returns.columns]
CORR_LABELS_X = {"sp500": "S&P 500 TR", "ftse100": "FTSE 100 TR", "gold": "Gold"}

# Compute pairwise correlation for all Liv-ex indices vs comparison assets
lx_rets  = returns[LIVEX_COLS].dropna(how="all")
cmp_rets = returns[CORR_ASSETS].dropna(how="all")
common_idx   = lx_rets.index.intersection(cmp_rets.index)

corr_matrix = (
    pd.concat([lx_rets.loc[common_idx], cmp_rets.loc[common_idx]], axis=1)
    .corr()
    .loc[LIVEX_COLS, CORR_ASSETS]
)

x_labels = [CORR_LABELS_X.get(c, c) for c in CORR_ASSETS]
y_labels  = [LABELS.get(c, c) for c in LIVEX_COLS]

fig, ax = plt.subplots(figsize=(7, max(4, len(LIVEX_COLS) * 0.70)), dpi=DPI)

sns.heatmap(
    corr_matrix,
    ax=ax,
    annot=True,
    fmt=".2f",
    cmap=sns.diverging_palette(220, 20, as_cmap=True),
    vmin=-0.6,
    vmax=0.6,
    center=0,
    linewidths=0.5,
    linecolor="white",
    xticklabels=x_labels,
    yticklabels=y_labels,
    annot_kws={"size": 9},
)

ax.set_title(
    "Correlation: Liv-ex Indices vs Global Assets\n"
    f"(Monthly log-returns, {common_idx.min().year}–{common_idx.max().year})",
    fontsize=13, color=MIDNIGHT, pad=12
)
ax.set_xlabel("Comparison Asset", fontsize=10)
ax.set_ylabel("Liv-ex Index", fontsize=10)
ax.tick_params(axis="x", rotation=0)
ax.tick_params(axis="y", rotation=0)

fig.text(
    0.5, -0.03,
    "Source: Liv-ex, Yahoo Finance. Monthly log-returns. © WineFi",
    ha="center", fontsize=8, color=WARM_GREY, style="italic"
)
plt.tight_layout()

out1 = IMAGES_DIR / "01_static_correlation_heatmap.png"
fig.savefig(out1, dpi=DPI, bbox_inches="tight", facecolor='white')
plt.close(fig)
print(f"Saved → {out1}")

Saved → /home/runner/work/report-for-me/report-for-me/projects/correlation-diversification/images/correlation/01_static_correlation_heatmap.png


### Chart 2 — Rolling Correlations: Liv-ex 100 vs S&P 500

12-month and 36-month rolling Pearson correlation between Liv-ex 100 and S&P 500 monthly log-returns. Crisis periods shaded.

In [6]:
if LX100_COL is None or "sp500" not in returns.columns:
    print(f"WARNING: LX100_COL={LX100_COL} or sp500 missing. Skipping Chart 2.")
else:
    pair_df  = returns[[LX100_COL, "sp500"]].dropna()
    roll_12  = pair_df[LX100_COL].rolling(12).corr(pair_df["sp500"])
    roll_36  = pair_df[LX100_COL].rolling(36).corr(pair_df["sp500"])

    fig, ax = plt.subplots(figsize=(12, 5), dpi=DPI)

    # Shade crisis bands
    for start, end, label in CRISES:
        if start < roll_36.index.max():
            ax.axvspan(start, min(end, roll_36.index.max()), alpha=0.12, color=CLARET, zorder=0)
            mid = start + (min(end, roll_36.index.max()) - start) / 2
            ax.text(
                mid, 0.82, label,
                ha="center", fontsize=7.5, color=CLARET,
                transform=ax.get_xaxis_transform(), style="italic"
            )

    ax.axhline(0, color=MIDNIGHT, linewidth=0.8, linestyle="--", alpha=0.6)
    ax.plot(roll_12.index, roll_12, color=CLARET,   linewidth=1.5, alpha=0.75, label="12m rolling")
    ax.plot(roll_36.index, roll_36, color=CHAMPAGNE, linewidth=2.0, label="36m rolling")

    ax.set_title(
        f"Liv-ex 100 vs S&P 500 TR: Rolling Correlation\n"
        "(Monthly log-returns; crisis periods shaded)",
        fontsize=13, color=MIDNIGHT
    )
    ax.set_xlabel("Date", fontsize=10)
    ax.set_ylabel("Pearson Correlation", fontsize=10)
    ax.set_ylim(-1.0, 1.0)
    ax.legend(fontsize=9, loc="lower left")

    fig.text(
        0.5, -0.03,
        "Source: Liv-ex, Yahoo Finance. © WineFi",
        ha="center", fontsize=8, color=WARM_GREY, style="italic"
    )
    plt.tight_layout()

    out2 = IMAGES_DIR / "02_rolling_correlations_vs_sp500.png"
    fig.savefig(out2, dpi=DPI, bbox_inches="tight", facecolor='white')
    plt.close(fig)
    print(f"Saved → {out2}")

Saved → /home/runner/work/report-for-me/report-for-me/projects/correlation-diversification/images/correlation/02_rolling_correlations_vs_sp500.png


### Chart 3 — Liv-ex 1000: 36m Rolling Correlation vs All Assets

36-month rolling correlation of Liv-ex 1000 vs S&P 500, FTSE 100, and Gold on one chart.
Uses Liv-ex 1000 as the reference index (broader than the Liv-ex 100) to show how the wider
fine wine market correlates with global assets. Falls back to Liv-ex 100 if LX1000 is unavailable.

In [7]:
# Use LX1000 as the reference index for Chart 3 (broader than LX100)
# Falls back to LX100 if LX1000 is unavailable
_chart3_ref_col = LX1000_COL if LX1000_COL is not None else LX100_COL

if _chart3_ref_col is None:
    print("WARNING: Neither LX1000_COL nor LX100_COL found. Skipping Chart 3.")
else:
    _ref_label = LABELS.get(_chart3_ref_col, _chart3_ref_col)
    fig, ax = plt.subplots(figsize=(12, 5), dpi=DPI)

    _cmp_map = [
        ("sp500",   "S&P 500 TR",  NAVY),
        ("ftse100", "FTSE 100 TR", FOREST),
        ("gold",    "Gold",        CHAMPAGNE),
    ]

    for col, lbl, clr in _cmp_map:
        if col not in returns.columns:
            continue
        pair_df = returns[[_chart3_ref_col, col]].dropna()
        roll_36 = pair_df[_chart3_ref_col].rolling(36).corr(pair_df[col])
        ax.plot(roll_36.index, roll_36, color=clr, linewidth=2.0, label=lbl)

    # Shade crises
    for start, end, label in CRISES:
        if start < returns.index.max():
            ax.axvspan(start, min(end, returns.index.max()), alpha=0.10, color=CLARET, zorder=0)

    ax.axhline(0, color=MIDNIGHT, linewidth=0.8, linestyle="--", alpha=0.6)

    ax.set_title(
        f"{_ref_label}: 36m Rolling Correlation vs Global Assets\n"
        "(Monthly log-returns; crisis bands shaded)",
        fontsize=13, color=MIDNIGHT
    )
    ax.set_xlabel("Date", fontsize=10)
    ax.set_ylabel("36m Rolling Pearson Correlation", fontsize=10)
    ax.set_ylim(-1.0, 1.0)
    ax.legend(fontsize=9, loc="lower left")

    fig.text(
        0.5, -0.03,
        "Source: Liv-ex, Yahoo Finance. © WineFi",
        ha="center", fontsize=8, color=WARM_GREY, style="italic"
    )
    plt.tight_layout()

    out3 = IMAGES_DIR / "03_lx1000_rolling_correlation_all_assets.png"
    fig.savefig(out3, dpi=DPI, bbox_inches="tight", facecolor='white')
    plt.close(fig)
    print(f"Saved → {out3}")

Saved → /home/runner/work/report-for-me/report-for-me/projects/correlation-diversification/images/correlation/03_lx1000_rolling_correlation_all_assets.png


### Chart 4 — GFC Drawdown Comparison

Two-panel chart for Jan 2007 – Dec 2010:
- **Top**: Indexed price performance (rebased to 100 at start)
- **Bottom**: Rolling drawdown from peak

In [8]:
GFC_START = "2007-01-01"
GFC_END   = "2010-12-31"

_gfc_cols = [
    (LX100_COL,   LABELS.get(LX100_COL, "Liv-ex 100"),  CLARET),
    ("sp500_gbp", "S&P 500 TR (GBP)",                    NAVY),
    ("ftse100",   "FTSE 100 TR",                         FOREST),
]
_gfc_cols = [(c, lbl, clr) for c, lbl, clr in _gfc_cols if c in combined.columns]

_gfc_price_cols = [c for c, _, _ in _gfc_cols]
gfc_prices = combined.loc[GFC_START:GFC_END, _gfc_price_cols].copy()
gfc_prices = gfc_prices.dropna(how="all")

# Rebase to 100 at first available date
base_date   = gfc_prices.index[gfc_prices.notna().all(axis=1)][0]
gfc_indexed = 100.0 * gfc_prices / gfc_prices.loc[base_date]


def rolling_drawdown(series):
    """Rolling drawdown from expanding peak, as percentage."""
    roll_max = series.expanding().max()
    return (series - roll_max) / roll_max * 100


fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(12, 8), dpi=DPI, sharex=True,
    gridspec_kw={"hspace": 0.06, "height_ratios": [1.2, 1]}
)

# ── Panel 1: Indexed performance ──
for col, lbl, clr in _gfc_cols:
    if col in gfc_indexed.columns:
        ax1.plot(gfc_indexed.index, gfc_indexed[col], color=clr, linewidth=2.0, label=lbl)

ax1.axhline(100, color=MIDNIGHT, linewidth=0.6, linestyle="--", alpha=0.5)
ax1.set_title(
    f"GFC Window ({base_date.strftime('%b %Y')} – Dec 2010): Indexed Performance & Drawdown",
    fontsize=13, color=MIDNIGHT, pad=10
)
ax1.set_ylabel(f"Indexed Price ({base_date.strftime('%b %Y')} = 100)", fontsize=10)
ax1.legend(fontsize=9, loc="lower left")

# ── Panel 2: Rolling drawdown ──
for col, lbl, clr in _gfc_cols:
    if col in gfc_prices.columns:
        dd = rolling_drawdown(gfc_prices[col])
        ax2.plot(dd.index, dd, color=clr, linewidth=2.0, label=lbl)

ax2.axhline(0, color=MIDNIGHT, linewidth=0.6, linestyle="--", alpha=0.5)
if LX100_COL in gfc_prices.columns:
    dd_lx = rolling_drawdown(gfc_prices[LX100_COL])
    ax2.fill_between(dd_lx.index, dd_lx, 0, alpha=0.08, color=CLARET)
ax2.set_ylabel("Drawdown from Peak (%)", fontsize=10)
ax2.set_xlabel("Date", fontsize=10)
ax2.legend(fontsize=9, loc="lower left")

fig.text(
    0.5, -0.01,
    "Source: Liv-ex, Yahoo Finance. GBP terms. S&P 500 TR converted via GBP/USD. © WineFi",
    ha="center", fontsize=8, color=WARM_GREY, style="italic"
)
plt.tight_layout()

out4 = IMAGES_DIR / "04_gfc_drawdown_comparison.png"
fig.savefig(out4, dpi=DPI, bbox_inches="tight", facecolor='white')
plt.close(fig)
print(f"Saved → {out4}")

Saved → /home/runner/work/report-for-me/report-for-me/projects/correlation-diversification/images/correlation/04_gfc_drawdown_comparison.png


### Chart 4b — COVID 2020 Multi-Window Comparison (Q1, H1, Full-Year)

> **Methodological note**: Fine wine is an illiquid asset — prices reflect settled
> transactions that can lag the market by weeks. Comparing fine wine performance over
> a single month (e.g. “March 2020”) is not valid for this asset class.
> We use three overlapping windows:
> - **Q1 2020** (January–March): captures the acute equity shock (3-month window)
> - **H1 2020** (January–June): captures shock + early recovery (6-month window)
> - **Full-year 2020** (January–December): complete calendar-year outcome (12-month window)

Three-panel bar chart: Liv-ex 100 vs S&P 500 (GBP) vs FTSE 100 for each window.

In [9]:
# Chart 4b: COVID 2020 multi-window comparison
# Methodological requirement: all windows >= 3 months for illiquid fine wine asset

COVID_WINDOWS = [
    ("Q1 2020",        "2020-01-01", "2020-03-31",  3),
    ("H1 2020",        "2020-01-01", "2020-06-30",  6),
    ("Full-Year 2020", "2020-01-01", "2020-12-31", 12),
]

_covid_assets = [
    (LX100_COL,   LABELS.get(LX100_COL, "Liv-ex 100"), WINEFI_COLOURS[0]),
    ("sp500_gbp", "S&P 500 TR (GBP)",                  WINEFI_COLOURS[4]),
    ("ftse100",   "FTSE 100 TR",                       WINEFI_COLOURS[6]),
]
_covid_assets = [(c, lbl, clr) for c, lbl, clr in _covid_assets if c in combined.columns]

if not _covid_assets or LX100_COL is None:
    print("WARNING: insufficient columns for Chart 4b. Skipping.")
else:
    fig, axes = plt.subplots(1, 3, figsize=(14, 6), dpi=DPI, sharey=False)
    fig.set_facecolor('white')
    fig.suptitle(
        "COVID 2020: Fine Wine vs Equities \u2014 Multi-Window Comparison\n"
        "All windows \u2265 3 months (fine wine is illiquid; single-month comparisons are not valid)",
        fontsize=12, color=MIDNIGHT, y=1.02
    )

    for ax, (window_label, start_str, end_str, n_months) in zip(axes, COVID_WINDOWS):
        ax.set_facecolor('white')
        start_ts = pd.Timestamp(start_str)
        end_ts   = pd.Timestamp(end_str)

        bar_labels  = []
        bar_values  = []
        bar_colours = []

        for col, lbl, clr in _covid_assets:
            series    = combined[col].dropna()
            start_val = series.asof(start_ts)
            end_val   = series.asof(end_ts)
            if pd.notna(start_val) and pd.notna(end_val) and start_val != 0:
                ret_pct = ((end_val / start_val) - 1) * 100
                bar_labels.append(lbl)
                bar_values.append(ret_pct)
                bar_colours.append(clr)

        bars = ax.bar(bar_labels, bar_values, color=bar_colours, width=0.55,
                      edgecolor="white", linewidth=0.8)

        for bar, val in zip(bars, bar_values):
            offset = -1.2 if val < 0 else 0.5
            va     = "top" if val < 0 else "bottom"
            txt_c  = "white" if abs(val) > 5 else MIDNIGHT
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                val + offset,
                f"{val:+.1f}%",
                ha="center", va=va, fontsize=9, color=txt_c, fontweight="bold"
            )

        ax.axhline(0, color=MIDNIGHT, linewidth=0.8)
        ax.set_title(f"{window_label}\n({n_months}-month window)", fontsize=11, color=MIDNIGHT)
        ax.set_ylabel("Total Return (%)", fontsize=9)
        ax.yaxis.grid(True, linestyle="--", alpha=0.4)
        ax.set_axisbelow(True)
        ax.tick_params(axis="x", labelsize=8)

    fig.text(
        0.5, -0.03,
        "Source: Liv-ex, Yahoo Finance. GBP terms. Returns measured from Jan 2020 month-end. "
        "Fine wine is illiquid \u2014 minimum 3-month windows required for valid comparisons. \u00a9 WineFi",
        ha="center", fontsize=8, color=WARM_GREY, style="italic"
    )
    plt.tight_layout()

    out4b = IMAGES_DIR / "04b_covid_multiwindow_comparison.png"
    fig.savefig(out4b, dpi=DPI, bbox_inches="tight", facecolor='white')
    plt.close(fig)
    print(f"Saved \u2192 {out4b}")


Saved → /home/runner/work/report-for-me/report-for-me/projects/correlation-diversification/images/correlation/04b_covid_multiwindow_comparison.png


### Chart 5 — Burgundy 150 vs S&P 500 (GBP) & FTSE 100, GFC Window

If Burgundy 150 is not in the data, a placeholder image is saved instead.

In [10]:
def _save_placeholder(path, message="Data Not Available"):
    """Save a clean placeholder PNG when data is absent."""
    fig, ax = plt.subplots(figsize=(10, 5), dpi=DPI)
    ax.set_facecolor('white')
    fig.set_facecolor('white')
    ax.text(0.5, 0.55, message,
            transform=ax.transAxes, ha="center", va="center",
            fontsize=20, color=WARM_GREY, fontweight="bold")
    ax.text(0.5, 0.40, path.name,
            transform=ax.transAxes, ha="center", va="center",
            fontsize=11, color=WARM_GREY, style="italic")
    ax.set_axis_off()
    plt.tight_layout()
    fig.savefig(path, dpi=DPI, bbox_inches="tight", facecolor='white')
    plt.close(fig)
    print(f"Placeholder saved → {path}")


out5 = IMAGES_DIR / "05_burgundy150_vs_sp500_ftse_2008.png"

if not BURGUNDY_AVAIL:
    print("Burgundy 150 not available — creating placeholder.")
    _save_placeholder(out5)
else:
    _b150_cols = [
        (BURGUNDY_COL, "Burgundy 150",   ROSE),
        ("sp500_gbp",  "S&P 500 TR (GBP)",  NAVY),
        ("ftse100",    "FTSE 100 TR",        FOREST),
    ]
    _b150_cols = [(c, lbl, clr) for c, lbl, clr in _b150_cols if c in combined.columns]

    gfc_b150 = combined.loc[GFC_START:GFC_END, [c for c, _, _ in _b150_cols]].copy()

    # Find first date where Burgundy 150 has data in the GFC window
    b150_first = gfc_b150[BURGUNDY_COL].first_valid_index()

    if b150_first is None:
        print("No Burgundy 150 data in GFC window — creating placeholder.")
        _save_placeholder(out5)
    else:
        gfc_b150_idx = 100.0 * gfc_b150 / gfc_b150.loc[b150_first]

        fig, ax = plt.subplots(figsize=(12, 5), dpi=DPI)

        for col, lbl, clr in _b150_cols:
            if col in gfc_b150_idx.columns:
                ax.plot(gfc_b150_idx.index, gfc_b150_idx[col],
                        color=clr, linewidth=2.0, label=lbl)

        ax.axhline(100, color=MIDNIGHT, linewidth=0.6, linestyle="--", alpha=0.5)
        ax.set_title(
            f"Burgundy 150 vs S&P 500 (GBP-Adjusted) & FTSE 100\n"
            f"GFC Window (rebased to 100 at {b150_first.strftime('%b %Y')})",
            fontsize=13, color=MIDNIGHT
        )
        ax.set_xlabel("Date", fontsize=10)
        ax.set_ylabel("Indexed Price (base = 100)", fontsize=10)
        ax.legend(fontsize=9, loc="upper left")

        fig.text(
            0.5, -0.03,
            "Source: Liv-ex, Yahoo Finance. GBP terms. S&P 500 TR converted via GBP/USD. © WineFi",
            ha="center", fontsize=8, color=WARM_GREY, style="italic"
        )
        plt.tight_layout()

        fig.savefig(out5, dpi=DPI, bbox_inches="tight", facecolor='white')
        plt.close(fig)
        print(f"Saved → {out5}")

Saved → /home/runner/work/report-for-me/report-for-me/projects/correlation-diversification/images/correlation/05_burgundy150_vs_sp500_ftse_2008.png


### Chart 6 — GFC Peak-to-Trough Returns (Bar Chart)

Bar chart: Burgundy 150, Liv-ex 100, S&P 500 (GBP), FTSE 100 returns from S&P 500 GFC peak (Oct 2007) to trough (Feb 2009).

If Burgundy 150 is absent, a placeholder is saved.

In [11]:
out6 = IMAGES_DIR / "06_burgundy150_gfc_bar_comparison.png"

if not BURGUNDY_AVAIL:
    print("Burgundy 150 not available — creating placeholder.")
    _save_placeholder(out6)
else:
    # GFC: S&P 500 peaked Oct 2007, troughed ~Feb 2009
    GFC_PEAK_DATE   = pd.Timestamp("2007-10-31")
    GFC_TROUGH_DATE = pd.Timestamp("2009-02-28")

    _bar_assets = [
        (BURGUNDY_COL, "Burgundy 150",  ROSE),
        (LX100_COL,    "Liv-ex 100",    CLARET),
        ("sp500_gbp",  "S&P 500 TR (GBP)", NAVY),
        ("ftse100",    "FTSE 100 TR",       FOREST),
    ]
    _bar_assets = [(c, lbl, clr) for c, lbl, clr in _bar_assets if c and c in combined.columns]

    gfc_returns_pct = {}
    for col, lbl, clr in _bar_assets:
        series = combined[col].dropna()
        peak_val   = series.asof(GFC_PEAK_DATE)
        trough_val = series.asof(GFC_TROUGH_DATE)
        if pd.notna(peak_val) and pd.notna(trough_val) and peak_val != 0:
            gfc_returns_pct[lbl] = ((trough_val / peak_val) - 1) * 100, clr

    if not gfc_returns_pct:
        print("Insufficient data for GFC bar chart — creating placeholder.")
        _save_placeholder(out6)
    else:
        _bar_labels  = list(gfc_returns_pct.keys())
        _bar_values  = [gfc_returns_pct[l][0] for l in _bar_labels]
        _bar_colours = [gfc_returns_pct[l][1] for l in _bar_labels]

        fig, ax = plt.subplots(figsize=(8, 5), dpi=DPI)

        bars = ax.bar(
            _bar_labels, _bar_values,
            color=_bar_colours, width=0.55,
            edgecolor="white", linewidth=0.8
        )

        # Annotate each bar with its return value
        for bar, val in zip(bars, _bar_values):
            offset = -1.5 if val < 0 else 0.5
            va     = "top" if val < 0 else "bottom"
            txt_c  = "white" if abs(val) > 8 else MIDNIGHT
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                val + offset,
                f"{val:.1f}%",
                ha="center", va=va,
                fontsize=11, color=txt_c, fontweight="bold"
            )

        ax.axhline(0, color=MIDNIGHT, linewidth=0.8)
        ax.set_title(
            "GFC Peak-to-Trough Returns\n"
            f"(S&P 500 peak {GFC_PEAK_DATE.strftime('%b %Y')} → trough {GFC_TROUGH_DATE.strftime('%b %Y')})",
            fontsize=13, color=MIDNIGHT
        )
        ax.set_ylabel("Total Return (%)", fontsize=10)
        ax.yaxis.grid(True, linestyle="--", alpha=0.4)
        ax.set_axisbelow(True)

        fig.text(
            0.5, -0.03,
            "Source: Liv-ex, Yahoo Finance. GBP terms. S&P 500 TR converted via GBP/USD. © WineFi",
            ha="center", fontsize=8, color=WARM_GREY, style="italic"
        )
        plt.tight_layout()

        fig.savefig(out6, dpi=DPI, bbox_inches="tight", facecolor='white')
        plt.close(fig)
        print(f"Saved → {out6}")

Saved → /home/runner/work/report-for-me/report-for-me/projects/correlation-diversification/images/correlation/06_burgundy150_gfc_bar_comparison.png


## 5. Output Assertions

Verify all 6 PNGs exist and are non-trivially sized.

In [12]:
EXPECTED_FILES = [
    "01_static_correlation_heatmap.png",
    "02_rolling_correlations_vs_sp500.png",
    "03_lx1000_rolling_correlation_all_assets.png",
    "04_gfc_drawdown_comparison.png",
    "04b_covid_multiwindow_comparison.png",
    "05_burgundy150_vs_sp500_ftse_2008.png",
    "06_burgundy150_gfc_bar_comparison.png",
]

errors = []
for fname in EXPECTED_FILES:
    fpath = IMAGES_DIR / fname
    if not fpath.exists():
        errors.append(f"MISSING: {fname}")
    elif fpath.stat().st_size < 5_000:
        errors.append(f"TOO SMALL (<5 KB): {fname}  ({fpath.stat().st_size} bytes)")
    else:
        print(f"OK  {fname}  ({fpath.stat().st_size / 1024:.0f} KB)")

if errors:
    for e in errors:
        print(f"FAIL: {e}")
    raise AssertionError("Output assertions failed — see above")
else:
    print(f"\nAll {len(EXPECTED_FILES)} charts verified in {IMAGES_DIR}")

OK  01_static_correlation_heatmap.png  (131 KB)
OK  02_rolling_correlations_vs_sp500.png  (166 KB)
OK  03_lx1000_rolling_correlation_all_assets.png  (158 KB)
OK  04_gfc_drawdown_comparison.png  (230 KB)
OK  04b_covid_multiwindow_comparison.png  (110 KB)
OK  05_burgundy150_vs_sp500_ftse_2008.png  (129 KB)
OK  06_burgundy150_gfc_bar_comparison.png  (59 KB)

All 7 charts verified in /home/runner/work/report-for-me/report-for-me/projects/correlation-diversification/images/correlation
